In [4]:
from pathlib import Path
import json, random, sys, time

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

projectRoot = Path.cwd().resolve()
if projectRoot.name == "notebook":
    projectRoot = projectRoot.parents[1]
if str(projectRoot) not in sys.path:
    sys.path.insert(0, str(projectRoot))

from src.wajib.lstm.LSTM import buildLSTMKeras, trainLSTMKeras, trainLSTMDataset, LSTMScratch
from src.wajib.shared.layers import EmbeddingLayer, DenseLayer
from src.wajib.shared.preprocessing import (
    loadFlickr8kCaptions, buildVocabulary, saveVocabulary, loadVocabulary,
 )
from src.wajib.shared.decoder import greedyDecode
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu

seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("GPU terdeteksi:", gpus)


GPU terdeteksi: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [5]:
featuresNpy = projectRoot / "src/wajib/weights/features/flickr8k_features.npy"
featuresIdx = projectRoot / "src/wajib/weights/features/flickr8k_index.json"
captionsFile = projectRoot / "data/flickr8k/captions.txt"
vocabPath = projectRoot / "src/wajib/weights/vocab.json"
weightsDir = projectRoot / "src/wajib/weights/lstm"
weightsDir.mkdir(parents=True, exist_ok=True)

embedDim = 256
maxLen = 30
epochs = 5
batchSize = 64
cnnFeatDim = 2048

variations = [
    (1, 128),
    (1, 512),
    (2, 128),
    (2, 512),
    (3, 128),
    (3, 512),
]


In [6]:
xCnnTrain, xTokTrain, yTrain = trainLSTMDataset(imageFeatures, trainCaps, vocab, maxLen)
xCnnVal, xTokVal, yVal = trainLSTMDataset(imageFeatures, valCaps, vocab, maxLen)

print(f"Data latih -> xCnn={xCnnTrain.shape}, xTok={xTokTrain.shape}, y={yTrain.shape}")
print(f"Data validasi -> xCnn={xCnnVal.shape}, xTok={xTokVal.shape}, y={yVal.shape}")

histories = {}
for numLayers, hiddenDim in variations:
    name = f"lstm_{numLayers}L_{hiddenDim}h"
    savePath = weightsDir / f"{name}.keras"

    print(f"\n{'='*50}\n  {name}\n{'='*50}")

    model = buildLSTMKeras(
        vocabSize=len(vocab),
        embedDim=embedDim,
        hiddenDim=hiddenDim,
        numLstmLayers=numLayers,
        cnnFeatureDim=cnnFeatDim,
    )
    model.summary()

    hist = trainLSTMKeras(
        model,
        xCnnTrain, xTokTrain, yTrain,
        xCnnVal, xTokVal, yVal,
        epochs=epochs,
        batchSize=batchSize,
        savePath=str(savePath),
    )
    histories[name] = hist
    print(f"  Tersimpan ke {savePath.name}")

print(f"\nSemua {len(variations)} variasi selesai.")


NameError: name 'imageFeatures' is not defined

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, (name, hist) in zip(axes.flatten(), histories.items()):
    ax.plot(hist['loss'], label='latih')
    ax.plot(hist['val_loss'], label='validasi')
    ax.set_title(name)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

plt.suptitle("Loss Pelatihan LSTM per Variasi", y=1.01)
plt.tight_layout()
plt.savefig(weightsDir / "lstm_training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"{'Model':<22} {'Val Loss Terbaik':>16} {'Epoch':>7}")
print('-' * 47)
for name, hist in histories.items():
    print(f"{name:<22} {min(hist['val_loss']):>16.4f} {len(hist['val_loss']):>7}")

bestName = min(histories, key=lambda n: min(histories[n]['val_loss']))
bestLayers = int(bestName.split('L')[0].split('_')[1])
bestHidden = int(bestName.split('h')[0].split('_')[2])
print(f"\nModel terbaik: {bestName} (layer={bestLayers}, hidden={bestHidden})")


In [ ]:
def loadScratchFromKeras(modelPath, numLayers):
    model = tf.keras.models.load_model(str(modelPath))

    embedLayer = EmbeddingLayer()
    embedLayer.loadWeights(model.get_layer('embedding'))

    projLayer = DenseLayer()
    projLayer.loadWeights(model.get_layer('cnn_proj'))

    outLayer = DenseLayer(activation='softmax')
    outLayer.loadWeights(model.get_layer('output'))

    lstmModel = LSTMScratch()
    lstmModel.loadWeights([model.get_layer(f'lstm_{i}') for i in range(numLayers)])

    return lstmModel, projLayer, embedLayer, outLayer

bestPath = weightsDir / f"{bestName}.keras"
lstmModel, projLayer, embedLayer, outLayer = loadScratchFromKeras(bestPath, bestLayers)

print("Contoh prediksi dari model LSTM terbaik:")
print("=" * 60)
for img in list(valCaps.keys())[:3]:
    feat = imageFeatures[img]
    caption = ' '.join(greedyDecode(lstmModel, projLayer, embedLayer, outLayer, feat, vocab))
    gt = valCaps[img][0]
    print(f"Ground truth : {gt}")
    print(f"Prediksi     : {caption}")
    print()


In [ ]:
results = {}

for numLayers, hiddenDim in variations:
    name = f"lstm_{numLayers}L_{hiddenDim}h"
    path = weightsDir / f"{name}.keras"
    lstmModel, projLayer, embedLayer, outLayer = loadScratchFromKeras(path, numLayers)

    bleuScores = []
    startTime = time.time()

    for img, caps in testCaps.items():
        if img not in imageFeatures:
            continue
        hyp = greedyDecode(lstmModel, projLayer, embedLayer, outLayer, imageFeatures[img], vocab, maxLen)
        refs = [cap.split() for cap in caps]
        bleuScores.append(sentence_bleu(refs, hyp, weights=(0.25, 0.25, 0.25, 0.25)))

    elapsed = time.time() - startTime
    results[name] = {
        'bleu4': np.mean(bleuScores),
        'time_s': elapsed,
        'layers': numLayers,
        'hidden': hiddenDim,
    }
    print(f"{name:<22}  BLEU-4={results[name]['bleu4']:.4f}  waktu={elapsed:.1f}s")


In [ ]:
bestTestModel = max(results, key=lambda n: results[n]['bleu4'])

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
models = sorted(results.keys())
bleus = [results[m]['bleu4'] for m in models]
colors = ['red' if m == bestTestModel else 'blue' for m in models]

ax.bar(range(len(models)), bleus, color=colors)
ax.set_xticks(range(len(models)))
ax.set_xticklabels(models, rotation=45, ha='right')
ax.set_ylabel('Skor BLEU-4')
ax.set_title('Skor BLEU-4 untuk Variasi LSTM di Data Uji')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(weightsDir / "lstm_bleu4_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n{'='*60}")
print("Ringkasan BLEU-4 LSTM:")
print(f"{'='*60}")
print(f"{'Model':<22} {'BLEU-4':>10} {'Waktu (detik)':>14}")
print('-' * 50)
for model in sorted(results.keys(), key=lambda m: results[m]['bleu4'], reverse=True):
    print(f"{model:<22} {results[model]['bleu4']:>10.4f} {results[model]['time_s']:>14.1f}")

print(f"\nTerbaik (BLEU-4 data uji): {bestTestModel}")
print(f"  BLEU-4={results[bestTestModel]['bleu4']:.4f}")
